# Mount Drive and install DuckDB

In [ ]:
# mount drive so I can read the CSVs from yesterday
from google.colab import drive
drive.mount('/content/drive')

# DuckDB is an in-memory SQL engine that runs inside Python - no server needed
!pip install duckdb -q

print("Drive mounted and DuckDB installed")


# Load all 9 CSVs into DuckDB

In [ ]:
import duckdb
import pandas as pd

# path to where the data is sitting in Drive
DATA_PATH = "/content/drive/MyDrive/PharmaFlow/pharmaflow_data"

# spin up an in-memory database
con = duckdb.connect(database=':memory:')

# the 9 tables we need to load
tables = [
    "patients", "products", "orders", "order_items",
    "prescriptions", "shipments", "returns",
    "fulfillment_centers", "insurance"
]

print("Loading tables into DuckDB...")
for t in tables:
    # read_csv_auto figures out the schema for me
    con.execute(f"""
        CREATE TABLE {t} AS
        SELECT * FROM read_csv_auto('{DATA_PATH}/{t}.csv')
    """)
    rows = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"   {t}: {rows:,} rows")

print("\nAll tables loaded. Ready for queries.")


# Test that the joins work

In [ ]:
# quick check - join the big tables together and pull 5 rows
# if this works, everything is connected properly
test_query = """
SELECT
    o.order_id,
    o.order_date,
    p.country,
    p.city,
    pr.category,
    oi.line_total_eur,
    s.carrier,
    s.on_time_delivery
FROM orders o
JOIN patients p           ON o.patient_id = p.patient_id
JOIN order_items oi       ON o.order_id = oi.order_id
JOIN products pr          ON oi.product_id = pr.product_id
LEFT JOIN shipments s     ON o.order_id = s.order_id
WHERE o.order_date >= '2024-01-01'
LIMIT 5
"""

result = con.execute(test_query).fetchdf()
print("Joins work. Sample data:")
result


# 18 business queries

In [ ]:
# storing all queries in a dict so I can loop through them at the end
queries = {}


# ====================
# SECTION A - REVENUE & SALES
# ====================

# Q1: total revenue by year - is the business growing?
queries["Q1_total_gmv_by_year"] = """
SELECT
    EXTRACT(YEAR FROM o.order_date) AS year,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.line_total_eur), 2) AS gmv_eur
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Delivered'
GROUP BY year
ORDER BY year;
"""

# Q2: monthly trend - looking for seasonality
queries["Q2_monthly_revenue_trend"] = """
SELECT
    DATE_TRUNC('month', o.order_date) AS month,
    ROUND(SUM(oi.line_total_eur), 2) AS revenue_eur,
    COUNT(DISTINCT o.order_id) AS orders
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Delivered'
GROUP BY month
ORDER BY month;
"""

# Q3: how much revenue comes from Rx vs OTC
queries["Q3_rx_vs_otc_revenue"] = """
SELECT
    CASE WHEN o.is_prescription_order THEN 'Rx (Prescription)' ELSE 'OTC' END AS order_type,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(oi.line_total_eur), 2) AS revenue_eur,
    ROUND(SUM(oi.line_total_eur) * 100.0 / SUM(SUM(oi.line_total_eur)) OVER (), 2) AS pct_of_total
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Delivered'
GROUP BY order_type
ORDER BY revenue_eur DESC;
"""

# Q4: which therapeutic categories drive the most revenue
queries["Q4_top_categories"] = """
SELECT
    p.category,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(oi.line_total_eur), 2) AS revenue_eur,
    ROUND(AVG(oi.line_total_eur), 2) AS avg_line_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p     ON oi.product_id = p.product_id
WHERE o.order_status = 'Delivered'
GROUP BY p.category
ORDER BY revenue_eur DESC;
"""

# Q5: does Web/Mobile/Phone produce different AOV
queries["Q5_aov_by_channel"] = """
SELECT
    o.channel,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(oi.line_total_eur), 2) AS revenue_eur,
    ROUND(SUM(oi.line_total_eur) / COUNT(DISTINCT o.order_id), 2) AS aov_eur
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Delivered'
GROUP BY o.channel
ORDER BY revenue_eur DESC;
"""


# ====================
# SECTION B - E-REZEPT
# ====================

# Q6: E-Rezept adoption by year - the big strategic story
queries["Q6_e_rezept_adoption_by_year"] = """
SELECT
    EXTRACT(YEAR FROM o.order_date) AS year,
    COUNT(*) AS rx_orders,
    SUM(CASE WHEN o.uses_e_rezept THEN 1 ELSE 0 END) AS e_rezept_orders,
    ROUND(SUM(CASE WHEN o.uses_e_rezept THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS e_rezept_adoption_pct
FROM orders o
WHERE o.is_prescription_order = TRUE
GROUP BY year
ORDER BY year;
"""

# Q7: does E-Rezept actually speed up fulfillment vs paper?
queries["Q7_e_rezept_processing_time"] = """
SELECT
    CASE WHEN o.uses_e_rezept THEN 'E-Rezept' ELSE 'Paper Prescription' END AS prescription_type,
    COUNT(*) AS shipments,
    ROUND(AVG(s.processing_hours), 2) AS avg_processing_hours,
    ROUND(MEDIAN(s.processing_hours), 2) AS median_processing_hours
FROM orders o
JOIN shipments s ON o.order_id = s.order_id
WHERE o.is_prescription_order = TRUE
GROUP BY prescription_type
ORDER BY avg_processing_hours;
"""

# Q8: which DACH countries are leading the digital shift
queries["Q8_e_rezept_by_country"] = """
SELECT
    p.country,
    COUNT(*) AS rx_orders,
    SUM(CASE WHEN o.uses_e_rezept THEN 1 ELSE 0 END) AS e_rezept_orders,
    ROUND(SUM(CASE WHEN o.uses_e_rezept THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS adoption_pct
FROM orders o
JOIN patients p ON o.patient_id = p.patient_id
WHERE o.is_prescription_order = TRUE
GROUP BY p.country
ORDER BY adoption_pct DESC;
"""


# ====================
# SECTION C - PATIENT BEHAVIOR
# ====================

# Q9: are chronic patients more valuable than acute ones?
queries["Q9_chronic_vs_acute_revenue"] = """
SELECT
    CASE WHEN p.has_chronic_condition THEN 'Chronic Patient' ELSE 'Acute Patient' END AS segment,
    COUNT(DISTINCT p.patient_id) AS patients,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(oi.line_total_eur), 2) AS revenue_eur,
    ROUND(SUM(oi.line_total_eur) / COUNT(DISTINCT p.patient_id), 2) AS revenue_per_patient
FROM patients p
JOIN orders o       ON p.patient_id = o.patient_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Delivered'
GROUP BY segment
ORDER BY revenue_eur DESC;
"""

# Q10: who are our top 20 most valuable patients
queries["Q10_top_20_patients_ltv"] = """
SELECT
    p.patient_id,
    p.country,
    p.city,
    p.has_chronic_condition,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.line_total_eur), 2) AS lifetime_value_eur
FROM patients p
JOIN orders o       ON p.patient_id = o.patient_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Delivered'
GROUP BY p.patient_id, p.country, p.city, p.has_chronic_condition
ORDER BY lifetime_value_eur DESC
LIMIT 20;
"""

# Q11: what % of patients reorder within 90 days of first purchase
queries["Q11_repeat_purchase_90d"] = """
WITH first_orders AS (
    SELECT patient_id, MIN(order_date) AS first_order_date
    FROM orders
    WHERE order_status = 'Delivered'
    GROUP BY patient_id
),
repeat_check AS (
    SELECT
        f.patient_id,
        f.first_order_date,
        MAX(CASE
            WHEN o.order_date > f.first_order_date
             AND o.order_date <= f.first_order_date + INTERVAL '90 days'
            THEN 1 ELSE 0
        END) AS made_repeat_purchase
    FROM first_orders f
    JOIN orders o ON f.patient_id = o.patient_id
    WHERE o.order_status = 'Delivered'
    GROUP BY f.patient_id, f.first_order_date
)
SELECT
    COUNT(*) AS total_patients,
    SUM(made_repeat_purchase) AS repeat_buyers,
    ROUND(SUM(made_repeat_purchase) * 100.0 / COUNT(*), 2) AS repeat_purchase_rate_pct
FROM repeat_check;
"""

# Q12: how is new patient signup trending month over month
queries["Q12_patient_acquisition_trend"] = """
SELECT
    DATE_TRUNC('month', signup_date) AS month,
    COUNT(*) AS new_patients
FROM patients
GROUP BY month
ORDER BY month;
"""


# ====================
# SECTION D - OPERATIONS & COLD CHAIN
# ====================

# Q13: which fulfillment center has the best on-time rate
queries["Q13_otif_by_fc"] = """
SELECT
    fc.fc_name,
    fc.country,
    COUNT(*) AS total_shipments,
    SUM(CASE WHEN s.on_time_delivery THEN 1 ELSE 0 END) AS on_time_shipments,
    ROUND(SUM(CASE WHEN s.on_time_delivery THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS otif_pct
FROM shipments s
JOIN orders o              ON s.order_id = o.order_id
JOIN fulfillment_centers fc ON o.fc_id = fc.fc_id
GROUP BY fc.fc_name, fc.country
ORDER BY otif_pct DESC;
"""

# Q14: cold chain vs standard shipments - cold chain is critical for insulin etc
queries["Q14_cold_chain_performance"] = """
SELECT
    CASE WHEN s.cold_chain_required THEN 'Cold Chain' ELSE 'Standard' END AS shipment_type,
    COUNT(*) AS total_shipments,
    ROUND(AVG(s.delivery_days), 2) AS avg_delivery_days,
    ROUND(SUM(CASE WHEN s.on_time_delivery THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS otif_pct,
    ROUND(AVG(s.shipment_cost_eur), 2) AS avg_cost_eur
FROM shipments s
GROUP BY shipment_type
ORDER BY shipment_type;
"""

# Q15: which carriers are reliable, which are letting us down
queries["Q15_carrier_performance"] = """
SELECT
    s.carrier,
    COUNT(*) AS shipments,
    ROUND(AVG(s.delivery_days), 2) AS avg_delivery_days,
    ROUND(SUM(CASE WHEN s.on_time_delivery THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS otif_pct,
    ROUND(AVG(s.shipment_cost_eur), 2) AS avg_cost_eur
FROM shipments s
GROUP BY s.carrier
ORDER BY otif_pct DESC;
"""

# Q16: do Rx orders process slower than OTC?
queries["Q16_processing_time_rx_vs_otc"] = """
SELECT
    CASE WHEN o.is_prescription_order THEN 'Rx Order' ELSE 'OTC Order' END AS order_type,
    COUNT(*) AS shipments,
    ROUND(AVG(s.processing_hours), 2) AS avg_processing_hours,
    ROUND(MIN(s.processing_hours), 2) AS min_hours,
    ROUND(MAX(s.processing_hours), 2) AS max_hours
FROM orders o
JOIN shipments s ON o.order_id = s.order_id
GROUP BY order_type
ORDER BY avg_processing_hours;
"""


# ====================
# SECTION E - RETURNS
# ====================

# Q17: which categories have the highest return rates
# Note: Rx is not returnable by law in DE so this is OTC only
queries["Q17_return_rate_by_category"] = """
WITH otc_orders_by_cat AS (
    SELECT
        p.category,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    WHERE o.is_prescription_order = FALSE
      AND o.order_status = 'Delivered'
    GROUP BY p.category
),
returns_by_cat AS (
    SELECT
        p.category,
        COUNT(DISTINCT r.return_id) AS total_returns
    FROM returns r
    JOIN order_items oi ON r.order_id = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    GROUP BY p.category
)
SELECT
    o.category,
    o.total_orders,
    COALESCE(r.total_returns, 0) AS total_returns,
    ROUND(COALESCE(r.total_returns, 0) * 100.0 / o.total_orders, 2) AS return_rate_pct
FROM otc_orders_by_cat o
LEFT JOIN returns_by_cat r ON o.category = r.category
ORDER BY return_rate_pct DESC;
"""

# Q18: why are customers returning items
queries["Q18_top_return_reasons"] = """
SELECT
    r.return_reason,
    COUNT(*) AS return_count,
    ROUND(SUM(r.refund_amount_eur), 2) AS total_refund_eur,
    ROUND(AVG(r.refund_amount_eur), 2) AS avg_refund_eur,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_returns
FROM returns r
GROUP BY r.return_reason
ORDER BY return_count DESC;
"""


# ====================
# Run all queries and save results
# ====================

import os
RESULTS_PATH = "/content/drive/MyDrive/PharmaFlow/sql_results"
os.makedirs(RESULTS_PATH, exist_ok=True)

print("Running 18 business queries...\n")

results = {}
for name, sql in queries.items():
    try:
        df = con.execute(sql).fetchdf()
        results[name] = df
        df.to_csv(f"{RESULTS_PATH}/{name}.csv", index=False)
        print(f"   {name}: {len(df)} rows -> saved")
    except Exception as e:
        print(f"   {name}: FAILED - {e}")

print(f"\nAll results saved to: {RESULTS_PATH}")
print(f"{len(results)} queries executed successfully")


In [ ]:
# print the key results so I can sanity check them
print("="*70)
print("HEADLINE BUSINESS METRICS")
print("="*70)

print("\nQ1 - Total GMV by Year:")
print(results["Q1_total_gmv_by_year"].to_string(index=False))

print("\nQ3 - Rx vs OTC Revenue Split:")
print(results["Q3_rx_vs_otc_revenue"].to_string(index=False))

print("\nQ4 - Top Therapeutic Categories:")
print(results["Q4_top_categories"].head(5).to_string(index=False))

print("\nQ6 - E-Rezept Adoption:")
print(results["Q6_e_rezept_adoption_by_year"].to_string(index=False))

print("\nQ13 - On-Time Delivery by Fulfillment Center:")
print(results["Q13_otif_by_fc"].to_string(index=False))

print("\nQ14 - Cold Chain Performance:")
print(results["Q14_cold_chain_performance"].to_string(index=False))

print("\nQ15 - Carrier Performance:")
print(results["Q15_carrier_performance"].to_string(index=False))

print("\nQ17 - Return Rate by Category:")
print(results["Q17_return_rate_by_category"].to_string(index=False))
